In [ ]:
# ==============================================================================
# 📓 Enterprise RAG Cookbook: 06_generation_and_prompt_engineering.ipynb
# ==============================================================================
# الهدف: تطبيق أساليب التوليد والهندسة الأوامرية المتقدمة (Generation & Prompting):
# 1. Production Prompt Engineering & Guardrails
# 2. Source Attribution & Citation Tracking
# 3. Structured RAG Output with Pydantic
# 4. Real-time Output Streaming (LCEL)
# ==============================================================================

# !pip install langchain-community langchain-core langchain-openai pydantic

import os
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI # يمكن استبدالها بـ ChatGroq أو ChatOllama

print("✅ تم استيراد مكتبات التوليد وPrompt Engineering بنجاح!")

<div dir="rtl">

## 1. تصميم الـ Production Prompt & Guardrails

الـ Prompt في أنظمة RAG المؤسسية يتكون من أجزاء محددة بعناية لمنع الـ Hallucinations:
1. **الذاتية والوظيفة (Role):** تحديد وظيفة النموذج كخبير ملتزم بالسياق.
2. **قواعد الأمان (Strict Guardrails):** ممنوع التكهّن أو استخدام معرفة خارجية إذا كان السياق غير كافٍ.
3. **هيكلة السياق (Formatted Context):** تمرين الـ Chunks مع معرفاتها (`[Doc ID]`) لتمكين التتبع.

</div>

In [ ]:
# ==============================================================================
# 1. Enterprise RAG Prompt Template Design
# ==============================================================================

SYSTEM_RAG_PROMPT = """You are an enterprise AI assistant for corporate knowledge retrieval.
Your primary goal is to answer user questions accurately based STRICTLY on the provided Context.

CRITICAL RULES:
1. Rely ONLY on the clear facts contained within the Context below.
2. Do NOT use any external knowledge or make assumptions not directly supported by the Context.
3. If the answer cannot be deduced from the Context, state clearly: "I do not have enough information in the provided context to answer this question."
4. Always maintain a professional, neutral, and concise tone.

--- CONTEXT ---
{context}
---------------
"""

rag_prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_RAG_PROMPT),
    ("human", "{question}")
])

print("✅ تم تجهيز الـ System Prompt الهيكلي للإنتاج!")

<div dir="rtl">

## 2. ميكانيزم تنسيق السياق واستخراج المصادر (Context Formatting)

في بيئات الإنتاج، لا يمرر الـ Document Object مباشرة، بل تحول قائمة المستندات المسترجعة إلى نص مهيكل يحتوي على الـ Metadata والـ Chunk IDs.

</div>

In [ ]:
# ==============================================================================
# 2. Helper function to format documents with Source Attributes
# ==============================================================================

def format_docs_with_sources(docs: List[Document]) -> str:
    """تحويل القطع المسترجعة لنص منظم بالـ Metadata والمصادر"""
    formatted = []
    for i, doc in enumerate(docs, 1):
        source_name = doc.metadata.get("source", "Unknown Source")
        chunk_id = doc.metadata.get("chunk_id", f"Chunk-{i}")
        formatted.append(f"[Source: {source_name} | ID: {chunk_id}]\n{doc.page_content}")
    return "\n\n".join(formatted)

# مستندات تجريبية كأنها قادمة من الـ Retriever
sample_docs = [
    Document(
        page_content="Annual leave policy grants 21 business days after 6 months of probation.",
        metadata={"source": "hr_policy_2026.pdf", "chunk_id": "hr-010"}
    ),
    Document(
        page_content="Health insurance covers outpatient care up to $3,000 annually.",
        metadata={"source": "benefits_guide.pdf", "chunk_id": "med-004"}
    )
]

print(format_docs_with_sources(sample_docs))

<div dir="rtl">

## 3. التوليد المنظم وحساب التوثيق (Structured RAG with Citations)

استخدام **Pydantic** لإجبار الـ LLM على إرجاع الإجابة مع قائمة المصادر والـ Citations المحددة المعتمد عليها، لتسليمها لـ Frontend جاهز.

</div>

In [ ]:
# ==============================================================================
# 3. Pydantic Model for Citation & Confidence Tracking
# ==============================================================================

class RAGResponseSchema(BaseModel):
    answer: str = Field(
        description="The direct answer to the user question based strictly on context."
    )
    citations: List[str] = Field(
        description="List of Chunk IDs or Source names cited in the answer (e.g., ['hr-010'])."
    )
    confidence_score: float = Field(
        description="Self-assessed confidence score between 0.0 and 1.0 based on context relevance."
    )
    has_sufficient_context: bool = Field(
        description="True if context had enough information, False otherwise."
    )

# إعداد الـ LLM الموجه بالـ Schema
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_rag_llm = llm.with_structured_output(RAGResponseSchema)

# بناء الـ Chain الكاملة باستخدام LCEL
structured_rag_chain = rag_prompt_template | structured_rag_llm

# تجربة تشغيل الـ Chain الحقيقية
def generate_structured_rag_answer(question: str, docs: List[Document]) -> RAGResponseSchema:
    context_str = format_docs_with_sources(docs)
    response = structured_rag_chain.invoke({
        "context": context_str,
        "question": question
    })
    return response

# ملاحظة للمذاكرة: الكود جاهز للاستدعاء الفعلي عند توفر الـ API Key
print("✅ تم بناء Structured Generation Chain بنجاح!")

<div dir="rtl">

## 4. البث التدفق المباشر (Real-Time RAG Streaming)

في التطبيقات الموجهة للمستخدم النهائي، يجب دعم الـ **Streaming** لإعادة التوكنز في الوقت الفعلي عبر **LCEL `stream()`**.

</div>

In [ ]:
# ==============================================================================
# 4. Real-time Streaming Response Pipeline (LCEL)
# ==============================================================================

# سلسلة البث النصي المباشر السريعة
streaming_rag_chain = rag_prompt_template | llm | StrOutputParser()

def stream_rag_response(question: str, docs: List[Document]):
    """دالة للبث المباشر الموجه للـ User Interface / FastAPIs"""
    context_str = format_docs_with_sources(docs)
    
    print(f"💬 Question: {question}\n")
    print("🤖 Answer: ", end="", flush=True)
    
    # استخدام .stream() لبث النتائج توكن بتوكن
    # for chunk in streaming_rag_chain.stream({"context": context_str, "question": question}):
    #     print(chunk, end="", flush=True)
    print("[Streaming Token Simulation Ready]")

# stream_rag_response("How many vacation days do I get?", sample_docs)

<div dir="rtl">

## 📝 ملخص معايير التوليد للإنتاج (Production Checklist)

1. **الموثوقية (Factuality):** اضبط الـ `temperature=0` دائماً في مهام الـ RAG للحد من الابتكار العشوائي.
2. **التتبع (Citations):** لا تكتفِ بالإجابة؛ مرر للنماذج معرفات الـ Chunks المرفقة واسترجعها بأسلوب **Pydantic Structured Output**.
3. **التجربة (UX):** استخدم الـ Streaming (`.stream()`) في واجهات المستخدم لأن وقت الاستجابة ينخفض انطباعياً بشكل كبير لدى المستخدم.

</div>